# SEMANTIC CONTENT-BASED ENGINE

## 1. Import necessary libaries

In [12]:
import os
import pickle
import torch
import warnings
import pandas as pd
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
warnings.filterwarnings('ignore')

## 2. Load Processed Data 

In [13]:
print("\n Loading enriched movies from Phase 1...")

processed_path = '../data/processed/enriched_movies.csv'

if not os.path.exists(processed_path):
    raise FileNotFoundError(f" File not found: {processed_path}\nPlease complete Phase 1 first!")

enriched_movies = pd.read_csv(processed_path)

print(f"\n Loaded {len(enriched_movies)} movies")
print(f"\n Columns: {list(enriched_movies.columns)}")
print("\nSample content_text:")
print(enriched_movies['content_text'].iloc[0])


 Loading enriched movies from Phase 1...

 Loaded 9742 movies

 Columns: ['movieId', 'title', 'genres', 'tmdbId', 'overview', 'poster_path', 'release_date', 'vote_average', 'tmdb_genres', 'content_text']

Sample content_text:
Toy Story (1995) | Adventure, Animation, Children, Comedy, Fantasy | Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences. | Family, Comedy, Animation, Adventure


## 3. Data Cleaning

In [19]:
enriched_movies.head(3)

,movieId,title,genres,tmdbId,overview,poster_path,release_date,vote_average,tmdb_genres,content_text
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,862.0,"Led by Woody, Andy's toys live happily in his ...",/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,1995-11-22,7.971,"['Family', 'Comedy', 'Animation', 'Adventure']","Toy Story (1995) | Adventure, Animation, Child..."
1,2,Jumanji (1995),Adventure|Children|Fantasy,8844.0,When siblings Judy and Peter discover an encha...,/vgpXmVaVyUL7GGiDeiK1mKEKzcX.jpg,1995-12-15,7.244,"['Adventure', 'Fantasy', 'Family']","Jumanji (1995) | Adventure, Children, Fantasy ..."
2,3,Grumpier Old Men (1995),Comedy|Romance,15602.0,A family wedding reignites the ancient feud be...,/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg,1995-12-22,6.500,"['Romance', 'Comedy']","Grumpier Old Men (1995) | Comedy, Romance | A ..."


### 3a. Checking Null Values

In [35]:
enriched_movies.shape

(9742, 10)

In [36]:
enriched_movies.isna().sum()

movieId           0
title             0
genres            0
tmdbId            8
overview        122
poster_path     125
release_date    121
vote_average    121
tmdb_genres       0
content_text      0
dtype: int64

## 4. Feature Engineering

### 4a. Model Configuration

In [41]:
# It is pretrained NLP model — most commonly used with the Sentence Transformers library.
# It converts text (like movie descriptions, titles, or tags) into numerical vectors (embeddings) so machines can understand similarity. 
    
model_name = 'all-MiniLM-L6-v2'

#  MPS (Apple Silicon acceleration) with fallback
device = 'mps' if torch.backends.mps.is_available() else 'cpu'

if device == 'mps':
    try:
        model = SentenceTransformer(model_name, device=device)
        print(f"Model loaded on MPS (Apple Silicon)")
    except Exception as e:
        print(f"MPS issue: {e}")
        print("Falling back to CPU...")
        device = 'cpu'
        model = SentenceTransformer(model_name, device=device)
else:
    model = SentenceTransformer(model_name, device=device)
    print(f"Model loaded on {device.upper()}")

print(f"Model: {model_name} | Dimension: 384 | Device: {device}")

Model loaded on MPS (Apple Silicon)
Model: all-MiniLM-L6-v2 | Dimension: 384 | Device: mps


### 4b. Generating Embeddings

In [42]:
# Generating Embeddings.
print("\n3. Generating semantic embeddings...")

# Use batching for efficiency and lower memory usage 
batch_size = 64   

# Get the content texts
texts = enriched_movies['content_text'].tolist()

# Generate embeddings in batches with progress bar
embeddings = []

for i in tqdm(range(0, len(texts), batch_size), desc="Computing embeddings"):
    batch = texts[i:i+batch_size]
    batch_embeddings = model.encode(
        batch,
        batch_size=batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True   
    )
    embeddings.append(batch_embeddings)

# Combine all batches
movie_embeddings = np.vstack(embeddings)

print(f"Embeddings generated successfully!")
print(f"Shape: {movie_embeddings.shape}  (movies × dimensions)")


3. Generating semantic embeddings...


Computing embeddings: 100%|███████████████████| 153/153 [02:06<00:00,  1.21it/s]

Embeddings generated successfully!
Shape: (9742, 384)  (movies × dimensions)


## 5. Save Embeddings & Model Info

In [43]:

print("\n4. Saving embeddings and metadata...")

embeddings_dir = '../artifacts/content'
os.makedirs(embeddings_dir, exist_ok=True)

# Save embeddings as .npy (fastest & most efficient format)
embeddings_path = os.path.join(embeddings_dir, 'movie_embeddings.npy')
np.save(embeddings_path, movie_embeddings)
print(f"Embeddings saved to: {embeddings_path}")

# Save movie IDs and titles for easy lookup later
metadata = {
    'movieId': enriched_movies['movieId'].tolist(),
    'title': enriched_movies['title'].tolist(),
    'model_name': model_name,
    'embedding_dim': movie_embeddings.shape[1]
}

metadata_path = os.path.join(embeddings_dir, 'embedding_metadata.pkl')
with open(metadata_path, 'wb') as f:
    pickle.dump(metadata, f)

print(f"Metadata saved to: {metadata_path}")


4. Saving embeddings and metadata...
Embeddings saved to: ../artifacts/content/movie_embeddings.npy
Metadata saved to: ../artifacts/content/embedding_metadata.pkl


## 6. Test Similarity Function

In [44]:
from sklearn.metrics.pairwise import cosine_similarity

def get_content_recommendations(movie_title, top_n=10):
    """Return top similar movies based on semantic content."""
    
    # Find the index of the input movie
    matches = enriched_movies[enriched_movies['title'].str.contains(movie_title, case=False, na=False)]
    
    if len(matches) == 0:
        return "Movie not found. Try a different title."
    
    idx = matches.index[0]
    query_embedding = movie_embeddings[idx]
    
    # Compute cosine similarity, fast with normalized embeddings.
    similarities = cosine_similarity([query_embedding], movie_embeddings)[0]
    
    # Get top N similar movies excluding itself.
    similar_indices = similarities.argsort()[::-1][1:top_n+1]
    
    results = []
    for i in similar_indices:
        results.append({
            'title': enriched_movies.iloc[i]['title'],
            'similarity_score': round(float(similarities[i]), 4),
            'overview': enriched_movies.iloc[i]['overview'][:200] + '...' if pd.notna(enriched_movies.iloc[i]['overview']) else ''
        })
    
    return pd.DataFrame(results)

## 7. Testing 

In [45]:
test_results = get_content_recommendations("Toy Story", top_n=5)
print(test_results[['title', 'similarity_score']])

                   title  similarity_score
0     Toy Story 2 (1999)            0.7444
1     Toy Story 3 (2010)            0.7409
2  Goofy Movie, A (1995)            0.5989
3        Toy, The (1982)            0.5366
4         Jumanji (1995)            0.5358
